In [ ]:
from pytorch_tabnet.tab_model import TabNetClassifier
import numpy as np

print("TabNet OK")

TabNet OK


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import random
import json

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import StratifiedGroupKFold, StratifiedShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    precision_score, recall_score, average_precision_score
)

from pytorch_tabnet.tab_model import TabNetClassifier


# =====================================
# 0. 固定隨機種子
# =====================================
SEED = 42
np.random.seed(SEED)
random.seed(SEED)


# =====================================
# 0.1 小工具
# =====================================
def safe_metric(func, y_true, y_score_or_pred, **kwargs):
    try:
        return func(y_true, y_score_or_pred, **kwargs)
    except ValueError:
        return np.nan


# =====================================
# 1. 讀資料
# =====================================
df = pd.read_csv("final_dataset_for_ml_FULL.csv", encoding="utf-8-sig")


# =====================================
# 1.5 反轉反向指標
# =====================================
reverse_map = {
    "llama_vagueness_score_1": "llama_vagueness_clarity_score_1",
    "llama_deflection_score_1": "llama_deflection_transparency_score_1"
}

for old_col, new_col in reverse_map.items():
    if old_col not in df.columns:
        raise ValueError(f"{old_col} 不存在於資料中，無法反轉")
    df[new_col] = 1 - df[old_col].clip(0, 1)


# =====================================
# 2. target / groups
# =====================================
if "label" not in df.columns:
    raise ValueError("label 不存在於資料中")
if "Company" not in df.columns:
    raise ValueError("Company 不存在於資料中")
if df["Company"].isna().any():
    raise ValueError("Company 欄位有缺值，groups 不能有 NaN")

y = df["label"].astype(int)
groups_all = df["Company"]


# =====================================
# 3. feature groups
# =====================================
semantic_cols = [
    "llama_specificity_score_1",
    "llama_evidence_substantiation_score_1",
    "llama_vagueness_clarity_score_1",
    "llama_commitment_score_1",
    "llama_temporal_credibility_score_1",
    "llama_deflection_transparency_score_1",
    "llama_comparability_score_1"
]

lexical_cols = [
    "llama_has_scope_1",
    "llama_has_sbti_1",
    "llama_has_material_1",
    "llama_has_kpi_1",
    "llama_has_percent_1"
]

financial_cols = [
    "SIZE",
    "ROA",
    "Leverage",
    "Cash_flow",
    "market_cap"
]


# =====================================
# 4. 檢查欄位
# =====================================
all_needed_cols = semantic_cols + lexical_cols + financial_cols + ["label", "Company"]
missing_cols = [col for col in all_needed_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"以下欄位不存在於資料中: {missing_cols}")


# =====================================
# 5. Ablation sets
# =====================================
feature_sets = {
    "M1: Semantic": semantic_cols,
    "M2: Lexical": lexical_cols,
    "M3: Financial": financial_cols,
    "M4: Semantic + Lexical": semantic_cols + lexical_cols,
    "M5: Semantic + Financial": semantic_cols + financial_cols,
    "M6: Semantic + Lexical + Financial": semantic_cols + lexical_cols + financial_cols
}


# =====================================
# 6. outer CV
# =====================================
outer_cv = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED)


# =====================================
# 7. TabNet configs（像 RNN/CNN 一樣）
# 你可以先跑 3 組輕量版
# =====================================
param_configs = [
    {
        "n_d": 8,
        "n_a": 8,
        "n_steps": 3,
        "gamma": 1.3,
        "lambda_sparse": 1e-4,
        "mask_type": "entmax",
        "lr": 0.02,
        "max_epochs": 50,
        "patience": 8,
        "batch_size": 512,
        "virtual_batch_size": 128,
        "weights": 1,
        "val_size": 0.2
    },
    {
        "n_d": 16,
        "n_a": 16,
        "n_steps": 3,
        "gamma": 1.3,
        "lambda_sparse": 1e-4,
        "mask_type": "entmax",
        "lr": 0.02,
        "max_epochs": 50,
        "patience": 8,
        "batch_size": 512,
        "virtual_batch_size": 128,
        "weights": 1,
        "val_size": 0.2
    },
    {
        "n_d": 16,
        "n_a": 16,
        "n_steps": 3,
        "gamma": 1.5,
        "lambda_sparse": 1e-4,
        "mask_type": "entmax",
        "lr": 0.02,
        "max_epochs": 50,
        "patience": 8,
        "batch_size": 512,
        "virtual_batch_size": 128,
        "weights": 1,
        "val_size": 0.2
    }
]


# =====================================
# 8. sklearn 風格 TabNet wrapper
# =====================================
class TabNetWrapper(BaseEstimator, ClassifierMixin):
    def __init__(
        self,
        n_d=16,
        n_a=16,
        n_steps=3,
        gamma=1.3,
        lambda_sparse=1e-4,
        mask_type="entmax",
        lr=0.02,
        max_epochs=50,
        patience=8,
        batch_size=512,
        virtual_batch_size=128,
        weights=1,
        seed=42,
        verbose=0,
        val_size=0.2,
    ):
        self.n_d = n_d
        self.n_a = n_a
        self.n_steps = n_steps
        self.gamma = gamma
        self.lambda_sparse = lambda_sparse
        self.mask_type = mask_type
        self.lr = lr
        self.max_epochs = max_epochs
        self.patience = patience
        self.batch_size = batch_size
        self.virtual_batch_size = virtual_batch_size
        self.weights = weights
        self.seed = seed
        self.verbose = verbose
        self.val_size = val_size
        self.model_ = None

    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y).reshape(-1)

        use_validation = True
        unique_classes, class_counts = np.unique(y, return_counts=True)

        if len(unique_classes) < 2 or np.min(class_counts) < 2:
            use_validation = False

        if use_validation:
            try:
                sss = StratifiedShuffleSplit(
                    n_splits=1,
                    test_size=self.val_size,
                    random_state=self.seed
                )
                train_idx, val_idx = next(sss.split(X, y))
                X_train_inner, X_val = X[train_idx], X[val_idx]
                y_train_inner, y_val = y[train_idx], y[val_idx]
            except Exception:
                use_validation = False

        if not use_validation:
            X_train_inner, y_train_inner = X, y
            X_val, y_val = X, y

        self.model_ = TabNetClassifier(
            n_d=self.n_d,
            n_a=self.n_a,
            n_steps=self.n_steps,
            gamma=self.gamma,
            lambda_sparse=self.lambda_sparse,
            mask_type=self.mask_type,
            optimizer_params={"lr": self.lr},
            seed=self.seed,
            verbose=self.verbose,
        )

        self.model_.fit(
            X_train=X_train_inner,
            y_train=y_train_inner,
            eval_set=[(X_val, y_val)],
            eval_name=["valid"],
            eval_metric=["auc"],
            max_epochs=self.max_epochs,
            patience=self.patience,
            batch_size=self.batch_size,
            virtual_batch_size=self.virtual_batch_size,
            weights=self.weights,
            drop_last=False,
        )

        self.classes_ = np.array([0, 1])
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=np.float32)
        return self.model_.predict(X)

    def predict_proba(self, X):
        X = np.asarray(X, dtype=np.float32)
        return self.model_.predict_proba(X)


# =====================================
# 9. 找最佳 threshold
# =====================================
def find_best_threshold(y_true, y_prob):
    thresholds = np.arange(0.10, 0.91, 0.01)
    best_threshold = 0.50
    best_score = -1

    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        score = f1_score(y_true, y_pred, zero_division=0)
        if score > best_score:
            best_score = score
            best_threshold = t

    return best_threshold, best_score


# =====================================
# 10. 單一 config 評估
# 回傳: summary + fold_df
# =====================================
def evaluate_single_config(X, y, groups, feature_name, config):
    fold_metrics = []
    best_thresholds = []

    print(f"\n=== {feature_name} | config={config} ===")

    for fold_idx, (train_idx, test_idx) in enumerate(
        outer_cv.split(X, y, groups=groups), start=1
    ):
        print(f"[{feature_name}] Fold {fold_idx} started")

        X_train_df = X.iloc[train_idx].copy()
        X_test_df = X.iloc[test_idx].copy()
        y_train = y.iloc[train_idx].copy()
        y_test = y.iloc[test_idx].copy()

        # imputer
        imputer = SimpleImputer(strategy="median")
        X_train = imputer.fit_transform(X_train_df)
        X_test = imputer.transform(X_test_df)

        # scaler
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # model
        model = TabNetWrapper(
            n_d=config["n_d"],
            n_a=config["n_a"],
            n_steps=config["n_steps"],
            gamma=config["gamma"],
            lambda_sparse=config["lambda_sparse"],
            mask_type=config["mask_type"],
            lr=config["lr"],
            max_epochs=config["max_epochs"],
            patience=config["patience"],
            batch_size=config["batch_size"],
            virtual_batch_size=config["virtual_batch_size"],
            weights=config["weights"],
            seed=SEED,
            verbose=0,
            val_size=config["val_size"]
        )

        # fit
        model.fit(X_train, y_train)

        # threshold tuning：直接用 train prob 找 best threshold
        train_prob = model.predict_proba(X_train)[:, 1]
        best_threshold, best_train_f1 = find_best_threshold(y_train, train_prob)
        best_thresholds.append(best_threshold)

        # test evaluation
        test_prob = model.predict_proba(X_test)[:, 1]
        y_pred = (test_prob >= best_threshold).astype(int)

        fold_accuracy = accuracy_score(y_test, y_pred)
        fold_f1 = f1_score(y_test, y_pred, zero_division=0)
        fold_precision = precision_score(y_test, y_pred, zero_division=0)
        fold_recall = recall_score(y_test, y_pred, zero_division=0)
        fold_roc_auc = safe_metric(roc_auc_score, y_test, test_prob)
        fold_pr_auc = safe_metric(average_precision_score, y_test, test_prob)

        fold_result = {
            "fold": fold_idx,
            "Model": "TabNet + Attention-like",
            "Feature_Set": feature_name,
            "Config": json.dumps(config, ensure_ascii=False),
            "Num_Features": X.shape[1],
            "accuracy": fold_accuracy,
            "f1": fold_f1,
            "roc_auc": fold_roc_auc,
            "precision": fold_precision,
            "recall": fold_recall,
            "average_precision": fold_pr_auc,
            "best_threshold": best_threshold,
            "train_best_f1": best_train_f1
        }
        fold_metrics.append(fold_result)

        roc_text = f"{fold_roc_auc:.4f}" if pd.notna(fold_roc_auc) else "nan"
        pr_text = f"{fold_pr_auc:.4f}" if pd.notna(fold_pr_auc) else "nan"

        print(
            f"[{feature_name}] Fold {fold_idx} done | "
            f"Threshold={best_threshold:.2f} | "
            f"F1={fold_f1:.4f} | "
            f"ROC_AUC={roc_text} | "
            f"PR_AUC={pr_text}"
        )

    metrics_df = pd.DataFrame(fold_metrics)

    summary = {
        "Model": "TabNet + Attention-like",
        "Feature_Set": feature_name,
        "Config": json.dumps(config, ensure_ascii=False),
        "Num_Features": X.shape[1],

        "Accuracy_mean": metrics_df["accuracy"].mean(),
        "Accuracy_std": metrics_df["accuracy"].std(),

        "F1_mean": metrics_df["f1"].mean(),
        "F1_std": metrics_df["f1"].std(),

        "ROC_AUC_mean": metrics_df["roc_auc"].mean(),
        "ROC_AUC_std": metrics_df["roc_auc"].std(),

        "Precision_mean": metrics_df["precision"].mean(),
        "Precision_std": metrics_df["precision"].std(),

        "Recall_mean": metrics_df["recall"].mean(),
        "Recall_std": metrics_df["recall"].std(),

        "PR_AUC_mean": metrics_df["average_precision"].mean(),
        "PR_AUC_std": metrics_df["average_precision"].std(),

        "Mean_Best_Threshold": np.mean(best_thresholds),
        "Threshold_std": np.std(best_thresholds)
    }

    return summary, metrics_df


# =====================================
# 11. 每個 feature set 跑所有 config
# =====================================
all_results = []
all_fold_results = []

for feature_name, cols in feature_sets.items():
    X = df[cols].copy()

    for config in param_configs:
        summary_result, fold_result_df = evaluate_single_config(
            X, y, groups_all, feature_name, config
        )
        all_results.append(summary_result)
        all_fold_results.append(fold_result_df)


# =====================================
# 12. 結果整理
# =====================================
results_df = pd.DataFrame(all_results)
folds_df = pd.concat(all_fold_results, axis=0, ignore_index=True)

numeric_cols = results_df.select_dtypes(include=[np.number]).columns
results_df[numeric_cols] = results_df[numeric_cols].round(4)

fold_numeric_cols = folds_df.select_dtypes(include=[np.number]).columns
folds_df[fold_numeric_cols] = folds_df[fold_numeric_cols].round(4)

print("\n===== All Results =====")
print(results_df)

results_df.to_csv(
    "llama_TabNet_configstyle_groupedCV_threshold_mean_std.csv",
    index=False,
    encoding="utf-8-sig"
)

folds_df.to_csv(
    "llama_TabNet_configstyle_groupedCV_threshold_fold_results.csv",
    index=False,
    encoding="utf-8-sig"
)


# =====================================
# 13. 每個 feature set 挑 F1 最佳 config
# =====================================
best_results_df = (
    results_df.sort_values(["Feature_Set", "F1_mean"], ascending=[True, False])
    .groupby("Feature_Set", as_index=False)
    .first()
)

best_numeric_cols = best_results_df.select_dtypes(include=[np.number]).columns
best_results_df[best_numeric_cols] = best_results_df[best_numeric_cols].round(4)

print("\n===== Best Config per Feature Set =====")
print(best_results_df)

best_results_df.to_csv(
    "llama_TabNet_best_per_featureset_mean_std.csv",
    index=False,
    encoding="utf-8-sig"
)


=== M1: Semantic | config={'n_d': 8, 'n_a': 8, 'n_steps': 3, 'gamma': 1.3, 'lambda_sparse': 0.0001, 'mask_type': 'entmax', 'lr': 0.02, 'max_epochs': 50, 'patience': 8, 'batch_size': 512, 'virtual_batch_size': 128, 'weights': 1, 'val_size': 0.2} ===
[M1: Semantic] Fold 1 started

Early stopping occurred at epoch 17 with best_epoch = 9 and best_valid_auc = 0.95912
[M1: Semantic] Fold 1 done | Threshold=0.66 | F1=0.7273 | ROC_AUC=0.9655 | PR_AUC=0.6792
[M1: Semantic] Fold 2 started

Early stopping occurred at epoch 9 with best_epoch = 1 and best_valid_auc = 0.96855
[M1: Semantic] Fold 2 done | Threshold=0.67 | F1=0.5714 | ROC_AUC=1.0000 | PR_AUC=1.0000
[M1: Semantic] Fold 3 started

Early stopping occurred at epoch 36 with best_epoch = 28 and best_valid_auc = 1.0
[M1: Semantic] Fold 3 done | Threshold=0.86 | F1=0.5714 | ROC_AUC=0.9375 | PR_AUC=0.8304
[M1: Semantic] Fold 4 started

Early stopping occurred at epoch 37 with best_epoch = 29 and best_valid_auc = 1.0
[M1: Semantic] Fold 4 done

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import random
import json

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import StratifiedGroupKFold, StratifiedShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    precision_score, recall_score, average_precision_score
)

from pytorch_tabnet.tab_model import TabNetClassifier


# =====================================
# 0. 固定隨機種子
# =====================================
SEED = 42
np.random.seed(SEED)
random.seed(SEED)


# =====================================
# 0.1 小工具
# =====================================
def safe_metric(func, y_true, y_score_or_pred, **kwargs):
    try:
        return func(y_true, y_score_or_pred, **kwargs)
    except ValueError:
        return np.nan


# =====================================
# 1. 讀資料
# =====================================
df = pd.read_csv("final_dataset_for_ml_FULL.csv", encoding="utf-8-sig")


# =====================================
# 1.5 反轉反向指標
# =====================================
reverse_map = {
    "chatgpt_vagueness_score_1": "chatgpt_vagueness_clarity_score_1",
    "chatgpt_deflection_score_1": "chatgpt_deflection_transparency_score_1"
}

for old_col, new_col in reverse_map.items():
    if old_col not in df.columns:
        raise ValueError(f"{old_col} 不存在於資料中，無法反轉")
    df[new_col] = 1 - df[old_col].clip(0, 1)


# =====================================
# 2. target / groups
# =====================================
if "label" not in df.columns:
    raise ValueError("label 不存在於資料中")
if "Company" not in df.columns:
    raise ValueError("Company 不存在於資料中")
if df["Company"].isna().any():
    raise ValueError("Company 欄位有缺值，groups 不能有 NaN")

y = df["label"].astype(int)
groups_all = df["Company"]


# =====================================
# 3. feature groups
# =====================================
semantic_cols = [
    "chatgpt_specificity_score_1",
    "chatgpt_evidence_substantiation_score_1",
    "chatgpt_vagueness_clarity_score_1",
    "chatgpt_commitment_score_1",
    "chatgpt_temporal_credibility_score_1",
    "chatgpt_deflection_transparency_score_1",
    "chatgpt_comparability_score_1"
]

lexical_cols = [
    "chatgpt_has_scope_1",
    "chatgpt_has_sbti_1",
    "chatgpt_has_material_1",
    "chatgpt_has_kpi_1",
    "chatgpt_has_percent_1"
]

financial_cols = [
    "SIZE",
    "ROA",
    "Leverage",
    "Cash_flow",
    "market_cap"
]


# =====================================
# 4. 檢查欄位
# =====================================
all_needed_cols = semantic_cols + lexical_cols + financial_cols + ["label", "Company"]
missing_cols = [col for col in all_needed_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"以下欄位不存在於資料中: {missing_cols}")


# =====================================
# 5. Ablation sets
# =====================================
feature_sets = {
    "M1: Semantic": semantic_cols,
    "M2: Lexical": lexical_cols,
    "M3: Financial": financial_cols,
    "M4: Semantic + Lexical": semantic_cols + lexical_cols,
    "M5: Semantic + Financial": semantic_cols + financial_cols,
    "M6: Semantic + Lexical + Financial": semantic_cols + lexical_cols + financial_cols
}


# =====================================
# 6. outer CV
# =====================================
outer_cv = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED)


# =====================================
# 7. TabNet configs（像 RNN/CNN 一樣）
# 你可以先跑 3 組輕量版
# =====================================
param_configs = [
    {
        "n_d": 8,
        "n_a": 8,
        "n_steps": 3,
        "gamma": 1.3,
        "lambda_sparse": 1e-4,
        "mask_type": "entmax",
        "lr": 0.02,
        "max_epochs": 50,
        "patience": 8,
        "batch_size": 512,
        "virtual_batch_size": 128,
        "weights": 1,
        "val_size": 0.2
    },
    {
        "n_d": 16,
        "n_a": 16,
        "n_steps": 3,
        "gamma": 1.3,
        "lambda_sparse": 1e-4,
        "mask_type": "entmax",
        "lr": 0.02,
        "max_epochs": 50,
        "patience": 8,
        "batch_size": 512,
        "virtual_batch_size": 128,
        "weights": 1,
        "val_size": 0.2
    },
    {
        "n_d": 16,
        "n_a": 16,
        "n_steps": 3,
        "gamma": 1.5,
        "lambda_sparse": 1e-4,
        "mask_type": "entmax",
        "lr": 0.02,
        "max_epochs": 50,
        "patience": 8,
        "batch_size": 512,
        "virtual_batch_size": 128,
        "weights": 1,
        "val_size": 0.2
    }
]


# =====================================
# 8. sklearn 風格 TabNet wrapper
# =====================================
class TabNetWrapper(BaseEstimator, ClassifierMixin):
    def __init__(
        self,
        n_d=16,
        n_a=16,
        n_steps=3,
        gamma=1.3,
        lambda_sparse=1e-4,
        mask_type="entmax",
        lr=0.02,
        max_epochs=50,
        patience=8,
        batch_size=512,
        virtual_batch_size=128,
        weights=1,
        seed=42,
        verbose=0,
        val_size=0.2,
    ):
        self.n_d = n_d
        self.n_a = n_a
        self.n_steps = n_steps
        self.gamma = gamma
        self.lambda_sparse = lambda_sparse
        self.mask_type = mask_type
        self.lr = lr
        self.max_epochs = max_epochs
        self.patience = patience
        self.batch_size = batch_size
        self.virtual_batch_size = virtual_batch_size
        self.weights = weights
        self.seed = seed
        self.verbose = verbose
        self.val_size = val_size
        self.model_ = None

    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y).reshape(-1)

        use_validation = True
        unique_classes, class_counts = np.unique(y, return_counts=True)

        if len(unique_classes) < 2 or np.min(class_counts) < 2:
            use_validation = False

        if use_validation:
            try:
                sss = StratifiedShuffleSplit(
                    n_splits=1,
                    test_size=self.val_size,
                    random_state=self.seed
                )
                train_idx, val_idx = next(sss.split(X, y))
                X_train_inner, X_val = X[train_idx], X[val_idx]
                y_train_inner, y_val = y[train_idx], y[val_idx]
            except Exception:
                use_validation = False

        if not use_validation:
            X_train_inner, y_train_inner = X, y
            X_val, y_val = X, y

        self.model_ = TabNetClassifier(
            n_d=self.n_d,
            n_a=self.n_a,
            n_steps=self.n_steps,
            gamma=self.gamma,
            lambda_sparse=self.lambda_sparse,
            mask_type=self.mask_type,
            optimizer_params={"lr": self.lr},
            seed=self.seed,
            verbose=self.verbose,
        )

        self.model_.fit(
            X_train=X_train_inner,
            y_train=y_train_inner,
            eval_set=[(X_val, y_val)],
            eval_name=["valid"],
            eval_metric=["auc"],
            max_epochs=self.max_epochs,
            patience=self.patience,
            batch_size=self.batch_size,
            virtual_batch_size=self.virtual_batch_size,
            weights=self.weights,
            drop_last=False,
        )

        self.classes_ = np.array([0, 1])
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=np.float32)
        return self.model_.predict(X)

    def predict_proba(self, X):
        X = np.asarray(X, dtype=np.float32)
        return self.model_.predict_proba(X)


# =====================================
# 9. 找最佳 threshold
# =====================================
def find_best_threshold(y_true, y_prob):
    thresholds = np.arange(0.10, 0.91, 0.01)
    best_threshold = 0.50
    best_score = -1

    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        score = f1_score(y_true, y_pred, zero_division=0)
        if score > best_score:
            best_score = score
            best_threshold = t

    return best_threshold, best_score


# =====================================
# 10. 單一 config 評估
# 回傳: summary + fold_df
# =====================================
def evaluate_single_config(X, y, groups, feature_name, config):
    fold_metrics = []
    best_thresholds = []

    print(f"\n=== {feature_name} | config={config} ===")

    for fold_idx, (train_idx, test_idx) in enumerate(
        outer_cv.split(X, y, groups=groups), start=1
    ):
        print(f"[{feature_name}] Fold {fold_idx} started")

        X_train_df = X.iloc[train_idx].copy()
        X_test_df = X.iloc[test_idx].copy()
        y_train = y.iloc[train_idx].copy()
        y_test = y.iloc[test_idx].copy()

        # imputer
        imputer = SimpleImputer(strategy="median")
        X_train = imputer.fit_transform(X_train_df)
        X_test = imputer.transform(X_test_df)

        # scaler
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # model
        model = TabNetWrapper(
            n_d=config["n_d"],
            n_a=config["n_a"],
            n_steps=config["n_steps"],
            gamma=config["gamma"],
            lambda_sparse=config["lambda_sparse"],
            mask_type=config["mask_type"],
            lr=config["lr"],
            max_epochs=config["max_epochs"],
            patience=config["patience"],
            batch_size=config["batch_size"],
            virtual_batch_size=config["virtual_batch_size"],
            weights=config["weights"],
            seed=SEED,
            verbose=0,
            val_size=config["val_size"]
        )

        # fit
        model.fit(X_train, y_train)

        # threshold tuning：直接用 train prob 找 best threshold
        train_prob = model.predict_proba(X_train)[:, 1]
        best_threshold, best_train_f1 = find_best_threshold(y_train, train_prob)
        best_thresholds.append(best_threshold)

        # test evaluation
        test_prob = model.predict_proba(X_test)[:, 1]
        y_pred = (test_prob >= best_threshold).astype(int)

        fold_accuracy = accuracy_score(y_test, y_pred)
        fold_f1 = f1_score(y_test, y_pred, zero_division=0)
        fold_precision = precision_score(y_test, y_pred, zero_division=0)
        fold_recall = recall_score(y_test, y_pred, zero_division=0)
        fold_roc_auc = safe_metric(roc_auc_score, y_test, test_prob)
        fold_pr_auc = safe_metric(average_precision_score, y_test, test_prob)

        fold_result = {
            "fold": fold_idx,
            "Model": "TabNet + Attention-like",
            "Feature_Set": feature_name,
            "Config": json.dumps(config, ensure_ascii=False),
            "Num_Features": X.shape[1],
            "accuracy": fold_accuracy,
            "f1": fold_f1,
            "roc_auc": fold_roc_auc,
            "precision": fold_precision,
            "recall": fold_recall,
            "average_precision": fold_pr_auc,
            "best_threshold": best_threshold,
            "train_best_f1": best_train_f1
        }
        fold_metrics.append(fold_result)

        roc_text = f"{fold_roc_auc:.4f}" if pd.notna(fold_roc_auc) else "nan"
        pr_text = f"{fold_pr_auc:.4f}" if pd.notna(fold_pr_auc) else "nan"

        print(
            f"[{feature_name}] Fold {fold_idx} done | "
            f"Threshold={best_threshold:.2f} | "
            f"F1={fold_f1:.4f} | "
            f"ROC_AUC={roc_text} | "
            f"PR_AUC={pr_text}"
        )

    metrics_df = pd.DataFrame(fold_metrics)

    summary = {
        "Model": "TabNet + Attention-like",
        "Feature_Set": feature_name,
        "Config": json.dumps(config, ensure_ascii=False),
        "Num_Features": X.shape[1],

        "Accuracy_mean": metrics_df["accuracy"].mean(),
        "Accuracy_std": metrics_df["accuracy"].std(),

        "F1_mean": metrics_df["f1"].mean(),
        "F1_std": metrics_df["f1"].std(),

        "ROC_AUC_mean": metrics_df["roc_auc"].mean(),
        "ROC_AUC_std": metrics_df["roc_auc"].std(),

        "Precision_mean": metrics_df["precision"].mean(),
        "Precision_std": metrics_df["precision"].std(),

        "Recall_mean": metrics_df["recall"].mean(),
        "Recall_std": metrics_df["recall"].std(),

        "PR_AUC_mean": metrics_df["average_precision"].mean(),
        "PR_AUC_std": metrics_df["average_precision"].std(),

        "Mean_Best_Threshold": np.mean(best_thresholds),
        "Threshold_std": np.std(best_thresholds)
    }

    return summary, metrics_df


# =====================================
# 11. 每個 feature set 跑所有 config
# =====================================
all_results = []
all_fold_results = []

for feature_name, cols in feature_sets.items():
    X = df[cols].copy()

    for config in param_configs:
        summary_result, fold_result_df = evaluate_single_config(
            X, y, groups_all, feature_name, config
        )
        all_results.append(summary_result)
        all_fold_results.append(fold_result_df)


# =====================================
# 12. 結果整理
# =====================================
results_df = pd.DataFrame(all_results)
folds_df = pd.concat(all_fold_results, axis=0, ignore_index=True)

numeric_cols = results_df.select_dtypes(include=[np.number]).columns
results_df[numeric_cols] = results_df[numeric_cols].round(4)

fold_numeric_cols = folds_df.select_dtypes(include=[np.number]).columns
folds_df[fold_numeric_cols] = folds_df[fold_numeric_cols].round(4)

print("\n===== All Results =====")
print(results_df)

results_df.to_csv(
    "chatgpt_TabNet_configstyle_groupedCV_threshold_mean_std.csv",
    index=False,
    encoding="utf-8-sig"
)

folds_df.to_csv(
    "chatgpt_TabNet_configstyle_groupedCV_threshold_fold_results.csv",
    index=False,
    encoding="utf-8-sig"
)


# =====================================
# 13. 每個 feature set 挑 F1 最佳 config
# =====================================
best_results_df = (
    results_df.sort_values(["Feature_Set", "F1_mean"], ascending=[True, False])
    .groupby("Feature_Set", as_index=False)
    .first()
)

best_numeric_cols = best_results_df.select_dtypes(include=[np.number]).columns
best_results_df[best_numeric_cols] = best_results_df[best_numeric_cols].round(4)

print("\n===== Best Config per Feature Set =====")
print(best_results_df)

best_results_df.to_csv(
    "chatgpt_TabNet_best_per_featureset_mean_std.csv",
    index=False,
    encoding="utf-8-sig"
)


=== M1: Semantic | config={'n_d': 8, 'n_a': 8, 'n_steps': 3, 'gamma': 1.3, 'lambda_sparse': 0.0001, 'mask_type': 'entmax', 'lr': 0.02, 'max_epochs': 50, 'patience': 8, 'batch_size': 512, 'virtual_batch_size': 128, 'weights': 1, 'val_size': 0.2} ===
[M1: Semantic] Fold 1 started

Early stopping occurred at epoch 17 with best_epoch = 9 and best_valid_auc = 0.99371
[M1: Semantic] Fold 1 done | Threshold=0.85 | F1=0.6000 | ROC_AUC=0.9741 | PR_AUC=0.8929
[M1: Semantic] Fold 2 started

Early stopping occurred at epoch 18 with best_epoch = 10 and best_valid_auc = 0.98742
[M1: Semantic] Fold 2 done | Threshold=0.90 | F1=0.5714 | ROC_AUC=0.9571 | PR_AUC=0.7000
[M1: Semantic] Fold 3 started

Early stopping occurred at epoch 10 with best_epoch = 2 and best_valid_auc = 0.96429
[M1: Semantic] Fold 3 done | Threshold=0.66 | F1=0.3333 | ROC_AUC=0.9375 | PR_AUC=0.6792
[M1: Semantic] Fold 4 started

Early stopping occurred at epoch 28 with best_epoch = 20 and best_valid_auc = 0.99394
[M1: Semantic] Fo